## The "root in container" & running as non-root

By default a container's main process runs as **root — UID 0 — inside its namespace**. People wave this away with "it's just root *inside* the container." It mostly isn't harmless. Root in the container can:

- modify any file in the container's filesystem, **including host files bind-mounted in as root**,
- use any **capability** the container still holds (a lot, in the default config),
- exploit a kernel **CVE** to escape into host context,
- read its own secrets, including anything passed as an env var.

And files it writes to a bind-mounted host dir are **owned by UID 0 on the host** (the module-04 ownership gotcha) — so "root in container" plus a host mount is effectively root over those host files. UID 0 in the container *is* UID 0 on the host; they share the user namespace.

**The fix is mundane and effective: don't run as root.** Most apps don't need to. Set the user in two places:

**In the Dockerfile** — create a dedicated system user *after* the root-needing steps:

```dockerfile
RUN useradd --system --uid 10001 --user-group --no-create-home appuser
USER appuser
```

Pick a **high UID** (e.g. `10001`) that won't collide with host UIDs; don't reuse `nobody` (65534). Put `USER` **after** package installs and `chown`s that need root.

**At run time** — `-u` overrides the image's `USER`:

```bash
docker run -u 10001:10001 myorg/api
docker run -u "$(id -u):$(id -g)" -v "$(pwd)":/work alpine   # files owned by you
```

That second form is the standard cure for the bind-mount ownership problem: run as your host UID and the files the container creates belong to you. Non-root is the single highest-value hardening step — everything later assumes it.